# Supplementary Figure 17 | GWAS-QTL colocalizations for restricted-phenotype models

**a.** Recovery of GWAS-QTL CoS (GWAS p < 5.7*10-9) discovered with an all-gene joint ColocBoost model for two restricted phenotype ColocBoost models: excluding novel lncRNA or protein-coding only (Methods). Bars are colored by whether the GWAS-QTL CoS was fully recovered, partially recovered, lost, or not eligible to be tested in the restricted phenotype model.

**b.** Discovery of QTL-GWAS CoS (GWAS p < 5.7*10-9) by four ColocBoost model configurations in whole blood: joint (all genes), joint (protein-coding only), single-phenotype (all genes), and single-phenotype (protein-coding). QTL-GWAS CoS were matched across models (Methods). Left bars show the number of QTL-GWAS CoS for each model, upper bars show the number of QTL-GWAS CoS for subsets of models indicated by the dots.

**c.** Cumulative distribution of normalized eQTL slope |β| at whole blood QTL-GWAS CoS (GWAS p < 5.7*10-9) lead variants for joint (all genes) and single-phenotype (all genes) ColocBoost models. |β| is the normalized eQTL slope at the lead variant. All eQTLs in the CoS contribute a |β|; sQTL effects are not shown. p-value from two-sided Kolmogorov-Smirnov test (*, p < 0.05; **, p < 0.01; ***, p < 0.001; ****, p < 0.0001).

**d.** Same as (c) but for the minimum |β| across genes within each CoS.


## Setup


In [ ]:
import re
from itertools import product
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import ks_2samp
from tqdm.auto import tqdm

plt.rc("axes", labelsize=6)
plt.rc("xtick", labelsize=6)
plt.rc("ytick", labelsize=6)
plt.rc("legend", fontsize=5)
plt.rc("legend", title_fontsize=6)
plt.rc("font", size=6)
plt.rc("figure", titlesize=7)
plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["ps.fonttype"] = 42
plt.rcParams["figure.figsize"] = (7.2, 5.0)
plt.rcParams["figure.dpi"] = 300
plt.rcParams["savefig.dpi"] = 300

np.random.seed(35934)

# Analysis and plot parameters
# Panel a recovery modes (Zenodo recovery_by_cos): only_v39 + only_protein_coding.
cos_modes = ["only_v39", "only_protein_coding"]
cos_mode_label = {
    "only_v39": "excluding\nnovel lncRNAs",
    "only_protein_coding": "protein-coding\nonly",
}
cos_status_order = ["not_eligible", "full", "partial", "lost"]
cos_status_color = {
    "not_eligible": "#D9D9D9",
    "lost": "#C44E52",
    "partial": "#8FBF8F",
    "full": "#2E7D32",
}
cos_status_label = {
    "not_eligible": "not eligible",
    "lost": "lost",
    "partial": "partial",
    "full": "full",
}
gws_neglog10p = float(-np.log10(5.7e-9))
sub_neglog10p = float(-np.log10(1e-5))
level_order = ["gws", "1e-5"]
mode_order = ["joint", "pc", "sg", "sg_pc"]
upset_modes = ["joint", "pc", "sg", "sg_pc"]
upset_mode_label = {
    "joint": "joint (all)",
    "pc": "joint (p.c.)",
    "sg": "single-phenotype (all)",
    "sg_pc": "single-phenotype (p.c.)",
}

ensg_stem = re.compile(r"(ENSG\d+)")
cdf_modes = ["joint", "sg"]
cdf_style = {
    "joint": {"color": "#222222", "ls": "-", "label": "joint"},
    "sg": {"color": "#6E6E6E", "ls": "--", "label": "single-phenotype"},
}
cdf_units = ["gene", "min"]
cdf_unit_label = {
    "gene": "per gene",
    "min": "min per CoS",
}
plot_level = "gws"
upset_level = "gws"
fig_width, fig_height, fig_dpi = 7.2, 5.0, 300

# Zenodo ColocBoost helpers
def tissue_from_cos_id(cid):
    s = str(cid)
    i = s.rfind("_")
    return s[:i] if i > 0 else s


def lead_from_lists(variants, vcp):
    vs = list(variants)
    vcps = list(vcp)
    ps = pd.to_numeric(pd.Series(vcps), errors="coerce")
    if len(ps) == len(vs) and ps.notna().any():
        return vs[int(ps.values.argmax())]
    return vs[0]


def cos_to_long(row):
    vs = list(row["variants"])
    ps = list(row["vcp"])
    return pd.DataFrame({
        "variant_id": vs,
        "vcp": ps[: len(vs)] + [float("nan")] * (len(vs) - len(ps)),
    })


def _coloc_list_len(val):
    if val is None:
        return 0
    if isinstance(val, float) and np.isnan(val):
        return 0
    if isinstance(val, np.ndarray):
        return len(val)
    if isinstance(val, (list, tuple, set)):
        return len(val)
    return 0


def _typeset(vals):
    if vals is None:
        return set()
    if isinstance(vals, float) and np.isnan(vals):
        return set()
    if isinstance(vals, np.ndarray):
        items = vals.tolist()
    elif isinstance(vals, (list, tuple, set)):
        items = list(vals)
    else:
        items = [vals]
    out = set()
    for v in items:
        if v is None:
            continue
        s = str(v).strip()
        if s and s not in ("nan", "None", "<NA>"):
            out.add(s)
    return out


def _member_qtl_class(types):
    t = _typeset(types)
    lnc = t & {"novel_lncRNA", "existing_lncRNA"}
    non = t - {"novel_lncRNA", "existing_lncRNA"}
    if not lnc:
        return "non-lncRNA"
    if non:
        return "Shared"
    return "lncRNA-specific (robust)"


def _locus_qtl_label(member_classes):
    labs = set(member_classes)
    if labs == {"lncRNA-specific (robust)"}:
        return "lncRNA-specific (robust)"
    if "Shared" in labs or ("non-lncRNA" in labs and "lncRNA-specific (robust)" in labs):
        return "Shared"
    return "non-lncRNA"


def load_main_colocboost(colocboost_dir, *, cols=None):
    frames = []
    for path in sorted(colocboost_dir.glob("*.ColocBoost.parquet")):
        if path.name.startswith("cross_"):
            continue
        tissue = path.name.replace(".ColocBoost.parquet", "")
        df = pd.read_parquet(path, columns=cols)
        df["tissue"] = tissue
        frames.append(df)
    if not frames:
        return pd.DataFrame()
    return pd.concat(frames, ignore_index=True)


def build_cross_tissue_members(cross_tissue_loci_path, colocboost_dir, *, include_ucos=False):
    loci = pd.read_parquet(cross_tissue_loci_path)
    all_cos = load_main_colocboost(colocboost_dir).set_index("cos_id", drop=False)
    memb = loci.explode("member_cos_ids").rename(columns={"member_cos_ids": "cos_id"})
    memb["cos_id"] = memb["cos_id"].astype(str)
    cos_rows = memb[["locus_id", "cos_id"]].merge(
        all_cos.reset_index(drop=True), on="cos_id", how="inner"
    )
    cos_rows["lead_variant_id"] = [
        lead_from_lists(v, p) for v, p in zip(cos_rows["variants"], cos_rows["vcp"])
    ]
    if "phenotype_id" not in cos_rows.columns:
        cos_rows["phenotype_id"] = cos_rows["phenotype_ids"]

    qtl_by_locus = (
        cos_rows.groupby("locus_id")["gene_types_robust"]
        .agg(lambda s: _locus_qtl_label(_member_qtl_class(x) for x in s))
    )
    locus_summary = loci.copy()
    locus_summary["qtl_label"] = locus_summary["locus_id"].map(qtl_by_locus)

    if not include_ucos:
        return locus_summary, cos_rows

    member_ids = set(memb["cos_id"])
    has_gwas = all_cos["gwas_traits"].map(_coloc_list_len) > 0
    has_qtl = all_cos["gene_ids_robust"].map(_coloc_list_len) > 0
    ucos = all_cos.loc[
        ~all_cos.index.isin(member_ids) & (has_gwas ^ has_qtl)
    ].reset_index(drop=True)
    ucos = ucos.loc[has_gwas.reindex(ucos["cos_id"]).fillna(False).values].copy()
    ucos["locus_id"] = "ucos_" + ucos["cos_id"].astype(str)
    ucos["lead_variant_id"] = [
        lead_from_lists(v, p) for v, p in zip(ucos["variants"], ucos["vcp"])
    ]
    if "phenotype_id" not in ucos.columns:
        ucos["phenotype_id"] = ucos["phenotype_ids"]
    members = pd.concat([cos_rows, ucos], ignore_index=True)
    return locus_summary, members




def _wb_mode_from_cos_id(cos_id: str) -> str:
    cid = str(cos_id)
    if cid.startswith("joint_"):
        return "joint"
    if cid.startswith("protein_coding_"):
        return "pc"
    if cid.startswith("single_phenotype_"):
        return "sg"
    return ""


def load_wb_mode_tables(joint_path, pc_path, sg_path):
    tables = {}
    for mode, path, tissue in (
        ("joint", joint_path, "joint"),
        ("pc", pc_path, "pc"),
        ("sg", sg_path, "sg"),
    ):
        df = pd.read_parquet(path)
        df["tissue"] = tissue
        df["lead_variant_id"] = [
            lead_from_lists(v, p) for v, p in zip(df["variants"], df["vcp"])
        ]
        tables[mode] = df.set_index("cos_id", drop=False)
    return tables


def build_wb_mode_merge_long(loci_path, joint_path, pc_path, sg_path):
    loci = pd.read_parquet(loci_path)
    tables = load_wb_mode_tables(joint_path, pc_path, sg_path)
    memb = loci.explode("member_cos_ids").rename(columns={"member_cos_ids": "cos_id"})
    memb["cos_id"] = memb["cos_id"].astype(str)
    rows = []
    for r in memb.itertuples(index=False):
        mode = _wb_mode_from_cos_id(r.cos_id)
        if not mode or r.cos_id not in tables[mode].index:
            continue
        row = tables[mode].loc[r.cos_id].to_dict()
        row["locus_id"] = str(r.locus_id)
        rows.append(row)
    return pd.DataFrame(rows)



## Paths

In [ ]:
gencode_class = Path("path/to/gencode_v39_v47_gene_classification.txt")
joint_leads_file = Path("path/to/Whole_Blood.joint.ColocBoost.parquet")
pc_leads_file = Path("path/to/Whole_Blood.protein_coding.ColocBoost.parquet")
sg_leads_file = Path("path/to/Whole_Blood.single_phenotype.ColocBoost.parquet")
wb_loci_path = Path("path/to/cross_phenotype_set_loci.parquet")
eqtl_allpairs = Path("path/to/v11_eqtl_allpairs")

# Zenodo coloc recovery summary + main tissue ColocBoost tables
recovery_by_cos_path = Path("path/to/recovery_by_cos.parquet")
colocboost_dir = Path("path/to/coloc_main")  # {Tissue}.ColocBoost.parquet
loci_path = Path("path/to/cross_tissue_loci.parquet")

fig_dir = Path("figures")


## Load data


### GENCODE gene classification


In [ ]:
gencode_genes = pd.read_table(gencode_class, low_memory=False)
gencode_genes = gencode_genes[gencode_genes["gene_type_grouped"] != "artifact"]
protein_coding_ids = set(
    gencode_genes.loc[gencode_genes["gene_type_grouped"] == "protein_coding", "gene_id_stable"]
)
print(f"Protein-coding genes: {len(protein_coding_ids):,}")

### Whole Blood joint leads, all genes


In [ ]:
lead_cols = [
    "cos_id",
    "gene_ids_robust",
    "phenotype_ids",
    "gwas_gws_traits",
    "cos_max_pval_nominal_phenotypes",
    "gene_types_robust",
    "variants",
    "vcp",
    "cos_npc",
    "gwas_traits",
]

def _load_wb_leads(path):
    df = pd.read_parquet(path, columns=lead_cols)
    df["lead_variant_id"] = [
        lead_from_lists(v, p) for v, p in zip(df["variants"], df["vcp"])
    ]
    return df

joint_leads = _load_wb_leads(joint_leads_file)
pc_leads = _load_wb_leads(pc_leads_file)
sg_leads = _load_wb_leads(sg_leads_file)
print(
    f"WB ColocBoost leads: joint={len(joint_leads):,}; pc={len(pc_leads):,}; "
    f"single_phenotype={len(sg_leads):,}"
)


### Whole Blood joint leads, protein-coding only


In [ ]:
pc_leads = pd.read_parquet(pc_leads_file, columns=lead_cols)
print(f"PC-only joint leads: {len(pc_leads):,}")

### Whole Blood single-gene leads


In [ ]:
sg_leads = pd.read_parquet(sg_leads_file, columns=lead_cols)
print(f"Single-gene leads: {len(sg_leads):,}")

### Whole Blood mode-merge long table (joint / PC / single-gene)


In [ ]:
merge_long = build_wb_mode_merge_long(wb_loci_path, joint_leads_file, pc_leads_file, sg_leads_file)
print(
    f"Mode-merge long: {len(merge_long):,} members; "
    f"{merge_long['locus_id'].nunique():,} loci; "
    f"tissues={sorted(merge_long['tissue'].astype(str).unique())}"
)


## Run analysis


### QTL–GWAS CoS at GWAS *p* cuts (per-model leads; no GWAS×LD filter)


In [ ]:
def gene_type_set(val):
    if val is None or (isinstance(val, float) and np.isnan(val)):
        return set()
    if isinstance(val, str):
        s = val.strip()
        if not s or s in {"nan", "None", "<NA>"}:
            return set()
        if s.startswith("[") and s.endswith("]"):
            s = s[1:-1]
        return {p.strip().strip("'\"") for p in s.split(",") if p.strip().strip("'\"")}
    try:
        return {str(x) for x in val if str(x) not in {"", "nan", "None"}}
    except TypeError:
        return {str(val)}


def parse_cs_max(val):
    if isinstance(val, (list, tuple, np.ndarray)):
        seq = list(val)
    else:
        s = str(val).strip()
        if not s or s in {"nan", "None", "<NA>"}:
            return []
        seq = s.split(",")
    out = []
    for x in seq:
        try:
            out.append(float(x))
        except (TypeError, ValueError):
            out.append(np.nan)
    return out


def gwas_traits_at_cut(phenotype_id, cs_max, cut):
    """GWAS traits in a CoS with CS-max -log10(p) > cut."""
    if phenotype_id is None or (isinstance(phenotype_id, float) and np.isnan(phenotype_id)):
        return []
    parts = [p.strip() for p in str(phenotype_id).split(";") if p.strip()]
    parsed = parse_cs_max(cs_max)
    out = []
    for i, part in enumerate(parts):
        if i >= len(parsed):
            break
        pl = part.lower()
        if "expression" in pl or "splicing" in pl:
            continue
        val = parsed[i]
        if np.isfinite(val) and float(val) > cut:
            out.append(part)
    return out


def qtl_cos(df):
    has_qtl = df["gene_ids_robust"].map(lambda v: _coloc_list_len(v) > 0)
    has_gwas = df["gwas_traits"].map(lambda v: _coloc_list_len(v) > 0)
    return df.loc[has_qtl & has_gwas].copy()


def cos_hits_at_cut(df, cut):
    """Boolean mask: QTL–GWAS CoS with ≥1 GWAS trait clearing the p cut."""
    return np.array(
        [
            len(gwas_traits_at_cut(pid, csmax, cut)) > 0
            for pid, csmax in zip(df["phenotype_ids"], df["cos_max_pval_nominal_phenotypes"])
        ],
        dtype=bool,
    )


leads_by_mode = {
    "joint": qtl_cos(joint_leads),
    "pc": qtl_cos(pc_leads),
    "sg": qtl_cos(sg_leads),
    "sg_pc": qtl_cos(sg_leads).loc[
        lambda d: d["gene_types_robust"].map(lambda v: "protein_coding" in gene_type_set(v))
    ].copy(),
}
# Zenodo ColocBoost has variants+vcp only; lead = variants[argmax(vcp)].
for _mode, _df in list(leads_by_mode.items()):
    if "lead_variant_id" not in _df.columns:
        _df = _df.copy()
        _df["lead_variant_id"] = [
            lead_from_lists(v, p) for v, p in zip(_df["variants"], _df["vcp"])
        ]
        leads_by_mode[_mode] = _df
cut_by_level = {"1e-5": sub_neglog10p, "gws": gws_neglog10p}

# Panels c/d: CoS with QTL + GWAS clearing the cut (same GWAS p rule as the caption;
# no Barbeira GWAS×LD denominator). Panel b uses mode-merged loci separately.
hit_by_key = {}
for level, cut in cut_by_level.items():
    for mode, df in leads_by_mode.items():
        hit = cos_hits_at_cut(df, cut)
        hit_by_key[(mode, level)] = hit
        print(f"{mode} {level}: {int(hit.sum()):,} QTL-GWAS CoS")


### eQTL phenotypes in the recovered CoS


In [ ]:
def expression_gene_ids(phenotype_id):
    genes = []
    if phenotype_id is None or (isinstance(phenotype_id, float) and np.isnan(phenotype_id)):
        return genes
    for part in str(phenotype_id).split(";"):
        part = part.strip()
        if ".expression_" not in part:
            continue
        genes.append(part.split(".expression_", 1)[1])
    return genes


def stem_of(gene_id):
    m = ensg_stem.search(str(gene_id))
    return m.group(1) if m else None


rows = []
for mode in tqdm(mode_order, desc="intersection eQTL"):
    df = leads_by_mode[mode]
    if "lead_variant_id" not in df.columns:
        df = df.copy()
        df["lead_variant_id"] = [
            lead_from_lists(v, p) for v, p in zip(df["variants"], df["vcp"])
        ]
        leads_by_mode[mode] = df
    for level in level_order:
        hit = hit_by_key[(mode, level)]
        for cos_id, phenotype_id, lead in zip(
            df.loc[hit, "cos_id"],
            df.loc[hit, "phenotype_ids"],
            df.loc[hit, "lead_variant_id"],
        ):
            lead_s = str(lead)
            if lead_s in {"", "nan", "None"} or ":" not in lead_s:
                continue
            chrom = lead_s.split(":", 1)[0]
            for gene_id in expression_gene_ids(phenotype_id):
                stem = stem_of(gene_id)
                if mode in {"pc", "sg_pc"} and stem not in protein_coding_ids:
                    continue
                rows.append({
                    "mode": mode,
                    "level": level,
                    "cos_id": str(cos_id),
                    "gene_id": gene_id,
                    "lead_variant_id": lead_s,
                    "chrom": chrom,
                })

intersection_df = pd.DataFrame(rows).drop_duplicates(
    ["mode", "level", "cos_id", "gene_id"]
)
print(f"Intersection eQTL rows: {len(intersection_df):,}")


### CoS-lead eQTL |β| (from Whole Blood allpairs)


In [ ]:
def allpairs_variant_ids(lead):
    parts = str(lead).split(":")
    if len(parts) < 4:
        return None, None
    chrom, pos, ref, alt = parts[:4]
    return f"{chrom}_{pos}_{ref}_{alt}_b38", f"{chrom}_{pos}_{alt}_{ref}_b38"


# Look up CoS-lead eQTL slopes from Whole Blood allpairs.
keys = intersection_df[["gene_id", "lead_variant_id", "chrom"]].drop_duplicates()
need = keys.copy()
slope_cache = pd.DataFrame(columns=["gene_id", "lead_variant_id", "slope", "slope_se", "af"])
print(f"Looking up slopes for {len(need):,} pairs")

query_rows = []
for gene_id, lead, chrom in need[["gene_id", "lead_variant_id", "chrom"]].itertuples(index=False):
    forward, flipped = allpairs_variant_ids(lead)
    if forward is None:
        continue
    query_rows.append(
        {"gene_id": gene_id, "lead_variant_id": lead, "chrom": chrom, "query_id": forward, "sign": 1}
    )
    query_rows.append(
        {"gene_id": gene_id, "lead_variant_id": lead, "chrom": chrom, "query_id": flipped, "sign": -1}
    )
query = pd.DataFrame(query_rows)
found_parts = []
for chrom, sub in tqdm(list(query.groupby("chrom")), desc="Whole Blood allpairs"):
    path = eqtl_allpairs / f"Whole_Blood.v11.allpairs.{chrom}.parquet"
    if not path.is_file():
        raise FileNotFoundError(f"Missing allpairs file: {path}")
    gene_list = sorted(set(sub["gene_id"]))
    variants = set(sub["query_id"])
    chunks = []
    step = 2000
    for start in range(0, len(gene_list), step):
        gene_chunk = gene_list[start : start + step]
        ap = pd.read_parquet(
            path,
            columns=["gene_id", "variant_id", "slope", "slope_se", "af"],
            filters=[("gene_id", "in", gene_chunk)],
        )
        ap = ap[ap["variant_id"].astype(str).isin(variants)]
        if len(ap):
            chunks.append(ap)
    if not chunks:
        continue
    ap = pd.concat(chunks, ignore_index=True).drop_duplicates(["gene_id", "variant_id"], keep="first")
    ap = ap.rename(columns={"variant_id": "query_id"})
    hit = sub.merge(ap, on=["gene_id", "query_id"], how="inner")
    if len(hit):
        found_parts.append(hit)
if found_parts:
    found = pd.concat(found_parts, ignore_index=True)
    found = found.sort_values("sign", ascending=False).drop_duplicates(
        ["gene_id", "lead_variant_id"], keep="first"
    )
    found["slope"] = found["slope"] * found["sign"]
    found["af"] = np.where(found["sign"] < 0, 1.0 - found["af"], found["af"])
    found = found[["gene_id", "lead_variant_id", "slope", "slope_se", "af"]]
else:
    found = pd.DataFrame(columns=["gene_id", "lead_variant_id", "slope", "slope_se", "af"])
new_rows = need.merge(found, on=["gene_id", "lead_variant_id"], how="left").drop(columns=["chrom"])
slope_cache = pd.concat([slope_cache, new_rows], ignore_index=True).drop_duplicates(
    ["gene_id", "lead_variant_id"], keep="last"
)

effect_df = intersection_df.merge(
    slope_cache[["gene_id", "lead_variant_id", "slope", "slope_se", "af"]],
    on=["gene_id", "lead_variant_id"],
    how="left",
)
effect_df["abs_slope"] = effect_df["slope"].abs()
n_found = int(effect_df["slope"].notna().sum())
print(f"CoS-lead slope found: {n_found:,} / {len(effect_df):,} ({100.0 * n_found / max(len(effect_df), 1):.1f}%)")


### UpSet membership at GWAS *p* < 5.7×10⁻⁹

In [ ]:
def _as_str_list(val):
    if val is None or (isinstance(val, float) and np.isnan(val)):
        return []
    if isinstance(val, (list, tuple, np.ndarray)):
        return [
            str(x) for x in val
            if x is not None and str(x) not in {"", "nan", "None", "<NA>"}
        ]
    s = str(val).strip()
    return [s] if s and s not in {"nan", "None", "<NA>"} else []


def _modes_for_row(row):
    mode = str(row.tissue)
    modes_here = {mode}
    if mode == "sg" and "protein_coding" in gene_type_set(getattr(row, "gene_types_robust", None)):
        modes_here.add("sg_pc")
    return modes_here


# Locus-level: GWAS trait × merged locus (not per gene).
# 1) Universe from traits with ≥1 genome-wide-significant variant (p < 5.7e-9).
# 2) Credit any Cos QTL at the same merged locus that carries the trait at all
#    (including CS-max below GWS), so shared-locus same-trait still counts.
merge_long = merge_long.copy()
merge_long["tissue"] = merge_long["tissue"].astype(str)
merge_long["locus_id"] = merge_long["locus_id"].astype(str)
is_cos = (
    merge_long["gene_ids_robust"].map(lambda v: _coloc_list_len(v) > 0)
    & merge_long["gwas_traits"].map(lambda v: _coloc_list_len(v) > 0)
)
has_qtl = merge_long["gene_ids_robust"].map(lambda v: len(_as_str_list(v)) > 0)
has_gws = merge_long["gwas_gws_traits"].map(lambda v: len(_as_str_list(v)) > 0)
locus_clear = merge_long.loc[is_cos & has_qtl & has_gws].copy()
locus_any = merge_long.loc[is_cos & has_qtl].copy()

trait_locus_modes = {}
for row in locus_clear.itertuples(index=False):
    traits = set(_as_str_list(getattr(row, "gwas_gws_traits", None)))
    if not traits:
        continue
    modes_here = _modes_for_row(row)
    for trait in traits:
        key = (trait, str(row.locus_id))
        trait_locus_modes.setdefault(key, set()).update(modes_here)

# Same merged locus + same trait present on Cos QTL → credit even if below GWS.
n_below_cut_credit = 0
for row in locus_any.itertuples(index=False):
    traits = (
        set(_as_str_list(getattr(row, "gwas_traits", None)))
        | set(_as_str_list(getattr(row, "gwas_gws_traits", None)))
    )
    if not traits:
        continue
    modes_here = _modes_for_row(row)
    for trait in traits:
        key = (trait, str(row.locus_id))
        if key not in trait_locus_modes:
            continue
        before = set(trait_locus_modes[key])
        trait_locus_modes[key].update(modes_here)
        if trait_locus_modes[key] != before:
            n_below_cut_credit += 1

locus_memb = pd.DataFrame(
    [
        {
            "trait": trait,
            "locus_id": lid,
            **{f"in_{m}": m in tissues for m in upset_modes},
            "n_models": sum(m in tissues for m in upset_modes),
            "combo": "+".join(m for m in upset_modes if m in tissues),
        }
        for (trait, lid), tissues in trait_locus_modes.items()
    ]
)


def upset_from_membership(memb, modes):
    flag_cols = [f"in_{m}" for m in modes]
    combos = [c for c in product([False, True], repeat=len(modes)) if any(c)]
    rows = []
    for combo in combos:
        mask = np.ones(len(memb), dtype=bool)
        for col, on in zip(flag_cols, combo):
            mask &= memb[col].astype(bool).to_numpy() == on
        n = int(mask.sum())
        if n:
            rows.append({"combo": combo, "count": n})
    upset = (
        pd.DataFrame(rows).sort_values("count", ascending=False).reset_index(drop=True)
        if rows else pd.DataFrame(columns=["combo", "count"])
    )
    set_sizes = np.array([int(memb[f"in_{m}"].sum()) for m in modes], dtype=float)
    return upset, set_sizes


def sort_upset_by_set_size(modes, upset_df, set_sizes):
    """Reorder set rows largest→smallest; remap intersection bitmasks."""
    order = np.argsort(-np.asarray(set_sizes, dtype=float))
    new_modes = [modes[i] for i in order]
    new_sizes = np.asarray(set_sizes, dtype=float)[order]
    if upset_df.empty:
        return new_modes, upset_df.copy(), new_sizes
    rows = []
    for combo, count in zip(upset_df["combo"], upset_df["count"]):
        combo = tuple(combo)
        rows.append({"combo": tuple(combo[i] for i in order), "count": count})
    new_upset = (
        pd.DataFrame(rows).sort_values("count", ascending=False).reset_index(drop=True)
    )
    return new_modes, new_upset, new_sizes


locus_upset, locus_set_sizes = upset_from_membership(locus_memb, upset_modes)
locus_modes_sorted, locus_upset, locus_set_sizes = sort_upset_by_set_size(
    upset_modes, locus_upset, locus_set_sizes
)
print(
    f"Locus UpSet: {len(locus_memb):,} GWAS×locus; "
    f"{len(locus_upset)} intersections; sizes="
    + ", ".join(f"{m}={int(n):,}" for m, n in zip(locus_modes_sorted, locus_set_sizes))
)
print(f"Same-locus same-trait below-cut credits added: {n_below_cut_credit:,}")
print("Locus combo counts:")
print(locus_memb["combo"].value_counts().to_string())


### UpSet set sizes at GWAS *p* < 1×10⁻⁵

In [ ]:
# p < 1e-5: any GWAS trait with CoS-max -log10(p) > 5
_SUB_NEGLOG10 = float(-np.log10(1e-5))


def _parse_neglog10_list(val):
    if val is None or (isinstance(val, float) and np.isnan(val)):
        return []
    out = []
    for tok in str(val).split(","):
        tok = tok.strip()
        if not tok or tok.lower() == "nan":
            out.append(np.nan)
            continue
        try:
            out.append(float(tok))
        except ValueError:
            out.append(np.nan)
    return out


def _gwas_traits_above_cut(phenotype_id, pval_nominal_phenotypes, cut) -> list:
    parts = [p.strip() for p in str(phenotype_id or "").split(";") if p.strip()]
    vals = _parse_neglog10_list(pval_nominal_phenotypes)
    out = []
    for i, part in enumerate(parts):
        if i >= len(vals):
            break
        pl = part.lower()
        if "expression" in pl or "splicing" in pl:
            continue
        v = vals[i]
        if np.isfinite(v) and float(v) > cut:
            out.append(part)
    return out


has_1e5 = np.array(
    [
        bool(_gwas_traits_above_cut(pid, pval, _SUB_NEGLOG10))
        for pid, pval in zip(
            merge_long["phenotype_ids"], merge_long["cos_max_pval_nominal_phenotypes"]
        )
    ],
    dtype=bool,
)
locus_clear_1e5 = merge_long.loc[is_cos.to_numpy() & has_qtl.to_numpy() & has_1e5]

# Universe: trait x locus with a CoS at p < 1e-5 in any model
trait_locus_modes_1e5 = {}
for row in locus_clear_1e5.itertuples(index=False):
    traits = set(_gwas_traits_above_cut(
        getattr(row, "phenotype_ids", None),
        getattr(row, "cos_max_pval_nominal_phenotypes", None),
        _SUB_NEGLOG10,
    ))
    modes_here = _modes_for_row(row)
    for trait in traits:
        trait_locus_modes_1e5.setdefault((trait, str(row.locus_id)), set()).update(modes_here)

# Credit any model carrying the same trait on a CoS QTL at the same locus
for row in locus_any.itertuples(index=False):
    traits = set(_as_str_list(getattr(row, "gwas_traits", None)))
    modes_here = _modes_for_row(row)
    for trait in traits:
        key = (trait, str(row.locus_id))
        if key in trait_locus_modes_1e5:
            trait_locus_modes_1e5[key].update(modes_here)

upset_sizes_1e5 = {
    m: sum(m in tissues for tissues in trait_locus_modes_1e5.values()) for m in upset_modes
}
print(f"GWAS x locus units at p < 1e-5: {len(trait_locus_modes_1e5):,}")


### CoS recovery summary (Zenodo `recovery_by_cos.parquet`)


In [ ]:
# Build panel-a stacked-bar counts from Zenodo recovery_by_cos.parquet.
# Columns: pc_only_recovery / v39_recovery ∈ {full, partial, lost}; NA = not eligible.
RECOVERY_COL = {
    "only_protein_coding": "pc_only_recovery",
    "only_v39": "v39_recovery",
}


def _as_type_list(val):
    if val is None:
        return []
    if isinstance(val, float) and np.isnan(val):
        return []
    if isinstance(val, np.ndarray):
        return list(val.tolist())
    if isinstance(val, (list, tuple, set)):
        return list(val)
    try:
        if pd.isna(val):
            return []
    except (TypeError, ValueError):
        pass
    s = str(val).strip()
    if not s or s in {"nan", "None", "<NA>"}:
        return []
    if ";" in s:
        return [p.strip() for p in s.split(";") if p.strip()]
    return [s]


def summarize_recovery_by_cos(recovery_df, modes):
    """Per-mode full / partial / lost / not_eligible counts over recovery_by_cos rows."""
    rows = []
    n_denom = len(recovery_df)
    for mode in modes:
        col = RECOVERY_COL[mode]
        s = recovery_df[col]
        n_full = int((s == "full").sum())
        n_partial = int((s == "partial").sum())
        n_lost = int((s == "lost").sum())
        n_not = int(s.isna().sum())
        rows.append(
            {
                "mode": mode,
                "n_denom": n_denom,
                "n_eligible": n_denom - n_not,
                "not_eligible": n_not,
                "full": n_full,
                "partial": n_partial,
                "lost": n_lost,
            }
        )
        print(
            f"{mode}: eligible={n_denom - n_not:,}; not_eligible={n_not:,}; "
            f"full={n_full:,}; partial={n_partial:,}; lost={n_lost:,}"
        )
    out = pd.DataFrame(rows)
    for st in cos_status_order:
        out[f"frac_{st}"] = out[st] / out["n_denom"].clip(lower=1)
    return out


### Compute — CoS recovery summary


In [ ]:
recovery_by_cos = pd.read_parquet(recovery_by_cos_path)
print(f"recovery_by_cos: {len(recovery_by_cos):,} CoS")
cos_summary = summarize_recovery_by_cos(recovery_by_cos, cos_modes)
print(cos_summary)


### CoS recovery summary (only_v39 + only_protein_coding)


In [ ]:
cos_summary = cos_summary.loc[cos_summary["mode"].isin(cos_modes)].reset_index(drop=True)
print(cos_summary)


## Make figure


In [ ]:
def sig_stars(p):
    if not np.isfinite(p):
        return "ns"
    if p < 1e-4:
        return "****"
    if p < 1e-3:
        return "***"
    if p < 1e-2:
        return "**"
    if p < 0.05:
        return "*"
    return "ns"


def format_pvalue(p):
    if not np.isfinite(p):
        return r"$p = \mathrm{NA}$"
    if p <= 0:
        return r"$p < 10^{-300}$"
    exp = int(np.floor(np.log10(p)))
    coef = p / (10.0 ** exp)
    if abs(coef - 1.0) < 1e-8:
        return rf"$p = 10^{{{exp}}}$"
    return rf"$p = {coef:.1f} \times 10^{{{exp}}}$"


cdf_x = {}
x_hi = 0.0
for mode in cdf_modes:
    sub = effect_df.loc[
        (effect_df["mode"] == mode)
        & (effect_df["level"] == plot_level)
        & effect_df["abs_slope"].notna(),
        ["cos_id", "abs_slope"],
    ]
    per_gene = np.sort(sub["abs_slope"].to_numpy(float))
    per_min = np.sort(sub.groupby("cos_id")["abs_slope"].min().to_numpy(float))
    cdf_x[(mode, "gene")] = per_gene
    cdf_x[(mode, "min")] = per_min
    for x in (per_gene, per_min):
        if len(x):
            x_hi = max(x_hi, float(np.quantile(x, 0.99)))

fig = plt.figure(figsize=(fig_width, fig_height), dpi=fig_dpi)
gs = fig.add_gridspec(2, 1, height_ratios=[1.5, 1], hspace=0.38)
gs_top = gs[0].subgridspec(1, 2, width_ratios=[.75, 2], wspace=0.28)
gs_bot = gs[1].subgridspec(1, 2, wspace=0.28)

# Panel a: CoS ablation recovery
ax_a = fig.add_subplot(gs_top[0, 0])
x_a = np.arange(len(cos_modes))
bottom = np.zeros(len(cos_modes), dtype=float)
for st in cos_status_order:
    heights = []
    for mode in cos_modes:
        row = cos_summary.loc[cos_summary["mode"] == mode]
        heights.append(float(row[f"frac_{st}"].iloc[0]) if len(row) else 0.0)
    heights = np.asarray(heights, dtype=float)
    ax_a.bar(
        x_a, heights, bottom=bottom, width=0.72,
        color=cos_status_color[st], edgecolor="k", linewidth=0.35,
        label=cos_status_label[st],
    )
    bottom += heights
ax_a.set_ylim(0, 1)
ax_a.set_yticks([0, 0.25, 0.5, 0.75, 1.0])
ax_a.set_yticklabels(["0", "25", "50", "75", "100"])
ax_a.set_ylabel("Joint GWAS-QTL CoS (%)", fontsize=5.5)
ax_a.set_xticks(x_a)
ax_a.set_xticklabels([cos_mode_label[m] for m in cos_modes], fontsize=4.5)
ax_a.tick_params(axis="y", labelsize=5)
ax_a.spines["top"].set_visible(False)
ax_a.spines["right"].set_visible(False)
ax_a.legend(
    frameon=False, loc="center left", bbox_to_anchor=(1.02, 0.5),
    fontsize=4.0, handlelength=0.9, labelspacing=0.15, borderpad=0.1,
    borderaxespad=0.0,
)

# Panel b: locus UpSet
_outer = gs_top[0, 1]
_upset_df = locus_upset
_set_sizes = locus_set_sizes
_modes = locus_modes_sorted
_ylab = "GWAS w/ QTL CoS"
_gs_u = _outer.subgridspec(
    2, 2, height_ratios=[0.66, 0.34], width_ratios=[0.28, 0.72], hspace=0.08, wspace=0.55,
)
ax_blank = fig.add_subplot(_gs_u[0, 0])
ax_bar = fig.add_subplot(_gs_u[0, 1])
ax_set = fig.add_subplot(_gs_u[1, 0])
ax_dot = fig.add_subplot(_gs_u[1, 1], sharex=ax_bar)
_combos = [tuple(c) for c in _upset_df["combo"]]
_counts = _upset_df["count"].to_numpy(dtype=float)
_x = np.arange(len(_combos))
_max_intersection = float(_counts.max()) if len(_counts) else 1.0
_dot_spacing = max(_max_intersection * 0.1, 1.0)
_y_pos = -np.arange(len(_modes)) * _dot_spacing
ax_bar.bar(_x, _counts, width=0.7, color="#9E9E9E", edgecolor="k", linewidth=0.4, zorder=2)
for xi, yi in zip(_x, _counts):
    ax_bar.text(xi, yi, f"{int(yi):,}", ha="center", va="bottom", fontsize=4.5)
ax_bar.set_ylabel(_ylab, fontsize=5.5)
ax_bar.set_xticks([])
ax_bar.spines["top"].set_visible(False)
ax_bar.spines["right"].set_visible(False)
ax_bar.spines["bottom"].set_visible(False)
ax_bar.tick_params(axis="x", bottom=False, labelbottom=False)
ax_bar.set_ylim(0, 200)
ax_bar.set_yticks([0, 50, 100, 150, 200])
_cat = np.arange(len(_modes))
ax_set.barh(
    y=_cat, width=_set_sizes, left=-_set_sizes,
    color="#9E9E9E", edgecolor="k", linewidth=0.4, height=0.55, zorder=2,
)
for yi, wi in zip(_cat, _set_sizes):
    ax_set.text(
        -float(wi), yi, f"{int(wi):,} ",
        ha="right", va="center", fontsize=4.5, color="0.15", zorder=3,
    )
ax_set.invert_yaxis()
ax_set.set_xlabel("GWAS w/ QTL CoS", fontsize=5)
ax_set.set_yticks(_cat)
ax_set.set_yticklabels([])
ax_set.tick_params(axis="y", left=False, length=0, labelsize=5)
ax_set.tick_params(axis="x", labelsize=5)
ax_set.spines["top"].set_visible(False)
ax_set.spines["left"].set_visible(False)
ax_set.spines["right"].set_visible(False)
_xmax = max(float(_set_sizes.max()), 1.0)
ax_set.set_xlim(-_xmax * 1.35, 0)
ax_set.xaxis.set_major_formatter(
    plt.FuncFormatter(lambda v, _pos: f"{int(round(abs(v)))}")
)
ax_dot.set_ylim(_y_pos.min() - _dot_spacing * 0.6, _dot_spacing * 0.6)
ax_dot.set_yticks(_y_pos)
ax_dot.set_yticklabels([upset_mode_label[m] for m in _modes], fontsize=4.5)
ax_dot.tick_params(axis="y", left=False, pad=-4, length=4, width=0.8)
for spine in ["top", "left", "right", "bottom"]:
    ax_dot.spines[spine].set_visible(False)
for idx, combo in enumerate(_combos):
    true_ys = []
    for j, present in enumerate(combo):
        ax_dot.scatter(idx, _y_pos[j], color=("black" if present else "lightgray"), s=28, zorder=3)
        if present:
            true_ys.append(_y_pos[j])
    if len(true_ys) >= 2:
        ax_dot.plot([idx] * len(true_ys), true_ys, color="black", linewidth=0.7, zorder=2)
ax_dot.set_xticks(_x)
ax_dot.set_xticklabels([""] * len(_x))
ax_dot.tick_params(axis="x", bottom=False, labelbottom=False)
ax_blank.axis("off")
ax_b = ax_bar

axes_cdf = []
panel_axes = {"a": ax_a, "b": ax_b}
for col, unit in enumerate(cdf_units):
    sharey = axes_cdf[0] if axes_cdf else None
    ax = fig.add_subplot(gs_bot[0, col], sharey=sharey)
    axes_cdf.append(ax)
    panel_axes["cd"[col]] = ax
    for mode in cdf_modes:
        x = cdf_x[(mode, unit)]
        y = np.arange(1, len(x) + 1) / max(len(x), 1)
        style = cdf_style[mode]
        ax.plot(
            x, y, drawstyle="steps-post", lw=1.0,
            color=style["color"], ls=style["ls"], label=style["label"],
        )
    ks = ks_2samp(cdf_x[("joint", unit)], cdf_x[("sg", unit)], alternative="two-sided")
    p = float(ks.pvalue)
    ax.legend(
        frameon=False, loc="lower right", fontsize=4.2,
        handlelength=1.4, labelspacing=0.2, borderpad=0.15,
    )
    ax.text(
        0.03, 0.97, f"KS {sig_stars(p)}\n{format_pvalue(p)}",
        transform=ax.transAxes, ha="left", va="top", fontsize=5, color="0.15",
    )
    ax.set_xlim(0, x_hi * 1.02)
    ax.set_ylim(0, 1)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.tick_params(labelsize=5)
    ax.set_ylabel("CDF", fontsize=5.5)
    ax.set_xlabel(f"{cdf_unit_label[unit]} " + r"eQTL $|\beta|$", fontsize=5.5)

fig.subplots_adjust(left=0.07, right=0.99, top=0.94, bottom=0.06)
_inv = fig.transFigure.inverted()
for letter, ax in panel_axes.items():
    _x, _y = _inv.transform(ax.transAxes.transform((-0.08, 1.12)))
    fig.text(_x, _y, letter, fontweight="bold", va="bottom", ha="right", fontsize=9)
plt.show()


## Save figure


In [ ]:
fig_dir.mkdir(parents=True, exist_ok=True)
fig.savefig(fig_dir / "supp_figure_17_coloc_single_vs_joint.pdf", bbox_inches="tight")
fig.savefig(fig_dir / "supp_figure_17_coloc_single_vs_joint.png", bbox_inches="tight")
print(f"Wrote {fig_dir / 'supp_figure_17_coloc_single_vs_joint.pdf'}")
print(f"Wrote {fig_dir / 'supp_figure_17_coloc_single_vs_joint.png'}")

## Print stats


In [ ]:
# Whole Blood: joint all-biotype vs restricted models (Results: ≥25% more)
_joint = upset_sizes_1e5["joint"]
_restricted = [upset_sizes_1e5[m] for m in ("pc", "sg", "sg_pc")]
_min_gain = 100.0 * (_joint / max(max(_restricted), 1) - 1.0)
print(f"Whole Blood joint all-biotype CoS: {_joint:,}")
print(
    f"vs restricted models (pc={upset_sizes_1e5['pc']:,}, "
    f"sg={upset_sizes_1e5['sg']:,}, sg_pc={upset_sizes_1e5['sg_pc']:,}): "
    f"min gain = {_min_gain:.0f}%"
)

# Cross-tissue protein-coding-only ablation: CoS with both a protein-coding
# gene and a lncRNA that are lost when lncRNAs are dropped from the joint model.
_lncrna = {"novel_lncRNA", "existing_lncRNA", "lncRNA"}


def _has_pc_and_lncrna(types):
    t = set(_as_type_list(types))
    return ("protein_coding" in t) and bool(t & _lncrna)


_main_frames = []
for _path in sorted(Path(colocboost_dir).glob("*.ColocBoost.parquet")):
    _main_frames.append(pd.read_parquet(_path, columns=["cos_id", "gene_types_robust"]))
if not _main_frames:
    raise FileNotFoundError(f"No *.ColocBoost.parquet under {colocboost_dir}")
_main = pd.concat(_main_frames, ignore_index=True)
_rec = pd.read_parquet(recovery_by_cos_path)
_merged = _main.merge(_rec[["cos_id", "pc_only_recovery"]], on="cos_id", how="inner")
_both = _merged.loc[_merged["gene_types_robust"].map(_has_pc_and_lncrna)]
n_both = len(_both)
n_lost = int((_both["pc_only_recovery"] == "lost").sum())
print(
    f"Cross-tissue both PC+lncRNA CoS: {n_both:,}; "
    f"not recovered by protein-coding-only joint: {n_lost:,} "
    f"({100.0 * n_lost / max(n_both, 1):.0f}%)"
)
